<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_G07_MULTISCALE_FAULT_DISCOVERY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS G07 - high-resolution multiscale fault discovery (code rev g07.1-r1)

Four coordinated systems: **A** surface (USGS 3DEP 1 m DEM geomorphology), **B** subsurface (REUSED G06 feature store, improved supervision), **C** candidate hypotheses + learned relevance, **D** DTI-aware exact-budget selection and gated export. The first milestone is the **DEM feasibility go / no-go**.

* Outputs: `MyDrive/GEMS_2/outputs/g07_multiscale_fault_discovery/<run_id>/` (run id = data-engineering identity; replaced outputs go to `history/`).
* **Nothing is uploaded.** The 0.40 leaderboard DTI is a target, never inferred from offline proxies. Docs: `docs/g07/`.

## 00 - Research objective and historical findings

**Hypothesis:** high-resolution surface morphology, improved fault supervision and independently generated subsurface hypotheses reveal structures W2 misses; a learned ranking + DTI-aware selection converts them into better predictions.

**Historical lessons (G04-G06), enforced in code:** W2 is the strongest reference; naive magnetic / gravity fusion lowered DTI; segment means destroy W2's pixel ranking (G07 keeps an exact-k W2 base); conservative corrections became inactive; G06 segment models had too few positives (G07 adds geometric positive augmentation of TRAINING truth); G06 exact-area evaluations were corrupted by ties (G07 selects exactly k pixels with label-free tie-breaking); geology-only models ranked withheld faults worse than W2 (G07 tests label design before model size).

| id | question |
|---|---|
| A0 | W2 exact-k reference: reproduce the baseline |
| A1 / A2 | does the DEM carry independent fault evidence / do scarps and terrain continuity help? |
| B1 / B2 | was label design limiting geology-only ML? / does a distance target improve localisation? |
| C1 | does spatial DEM context (U-Net) help? |
| C2 | do surface + subsurface candidate unions recover more systems at matched budget? |
| D1 / D2 | can a ranker separate true structures from false edges? / W2-preserving residual selection |
| E1 | continuous confidence vs binary top-area |

References (other protocols, never comparable): W2 V1 0.282538, W2 V3 0.130868, public 0.0942.

## 01 - CONFIG and RUN controls

First run = DEM feasibility (defaults). Later runs enable stages selectively; every validated stage is reused. Exact configurations: `docs/g07/COLAB_EXECUTION.md`.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                         # pin the pushed commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'output_subdir': 'outputs/g07_multiscale_fault_discovery',
    'seed': 2026,
    'dem_resolution_target_m': 1.0,        # validated against the real DEM headers (never assumed)
    'dem_tile_px': 1024,
    'dem_halo_px': 128,                    # minimum; the operator support raises it when needed
    'analysis_resolution_m': 100.0,
    'use_gpu': False,
    'reuse_g06_features': True,
    'reuse_w2_predictions': True,
    'strict_provenance': True,
    'save_figures': True,
    'auto_submit': False,                  # cannot be enabled
    'g06_run_dir': None,                   # e.g. '/content/drive/MyDrive/GEMS_2/outputs/g06_geological_feature_store/g06-xxxx'
    'run_synthetic_tests': True,
}
RUN = {
    'PREFLIGHT': True,
    'DEM_INVENTORY': True,
    'DEM_COVERAGE': True,
    'DEM_FEASIBILITY': True,
    'DEM_FEATURES': False,

    'BUILD_DATASETS': False,
    'TRAIN_SURFACE_CLASSIC': False,
    'TRAIN_SUBSURFACE': False,
    'TRAIN_SURFACE_UNET': False,
    'GENERATE_CANDIDATES': False,
    'TRAIN_CANDIDATE_RANKER': False,

    'EVALUATE': False,
    'FULL_INFERENCE': False,
    'EXPORT_SUBMISSION': False,

    'REPORT': True,
}
OVERRIDES = {}          # deep-merged into configs/g07/discovery.yaml, e.g. {'dem': {'download': {'enabled': True}}}
PRIVATE_REPO = False
EXPECT = {'code_rev': 'g07.1-r1', 'commit': None}

## 02 - Mount Drive and initialize repository

Restart the session before re-running if `gems_bench` is already imported.

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'g07_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab,m9]', 'pytest', 'scikit-image'], check=True)
sys.path.insert(0, CODE)
os.chdir(CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('HEAD', HEAD)

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
import gems_bench.g07 as G07
from gems_bench.g07 import config as GC
from gems_bench.g07.pipeline import G07Pipeline
assert G07.G07_CODE_REV == EXPECT['code_rev'], G07.G07_CODE_REV
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit'])
cfg = GC.load_config(overrides={'notebook': CONFIG, 'run': RUN, **OVERRIDES})
P = G07Pipeline(cfg, RUN, HEAD)
OUT = P.root

def show(rel, cols=None, n=25):
    p = OUT / rel
    if not p.exists():
        print('ABSENT:', rel); return None
    df = pd.read_csv(p) if p.suffix == '.csv' else pd.read_parquet(p)
    display(df[[c for c in (cols or df.columns) if c in df]].head(n)); return df

def js(rel, n=2000):
    d = P.ws.read_json(rel)
    print(json.dumps(d, indent=1, default=str)[:n] if d is not None else f'ABSENT: {rel}')

def figs(sub, limit=6):
    for f in sorted((OUT / 'figures' / sub).glob('*.png'))[:limit]:
        print(f.name); display(Image(str(f)))

print('run id', P.run_id, '| local', OUT, '| Drive', P.ws.drive_path())

## 03 - Validate repository commit and existing artifacts

**Input:** Drive (GeoDAWN raster, faults, template, P6 W2 maps, G06 store). **Operation:** synthetic tests (Colab), frozen-engine attach, artifact checks. **Output:** `dependencies/existing_artifacts.csv`, `missing_dependencies.json`. **Purpose:** know exactly which stages can run before spending compute.

In [ ]:
P.run_tests(); show('audit/tests.csv')
P.preflight()
show('dependencies/existing_artifacts.csv', ['artifact', 'mandatory', 'status', 'detail', 'blocks'], n=20)
js('dependencies/missing_dependencies.json')

## 04 - Verify official dataset and DEM source contracts

Exact band identifiers of the 19-band raster (G06 contract); DEM headers: resolution, CRS, vertical unit, nodata. A 100 m elevation band is **never** a high-resolution DEM (contract `assert_not_100m_substitute`). Licence: USGS 3DEP is public domain; competition eligibility of external data must be confirmed by you (`dem.competition_eligibility_verified`).

In [ ]:
d = P.ws.read_json('manifests/dependencies.json') or {}
print('grid', d.get('grid'))
P.dem_inventory()
show('dem_catalog/dem_inventory.csv', ['tile_id', 'status', 'res_m', 'crs', 'vertical_unit', 'vertical_unit_source', 'nodata', 'project', 'year', 'problems'], n=30)
show('dem_catalog/source_licenses.csv')

## 05 - DEM catalog and geographic coverage

Coverage is measured on the **eligible fault-discovery region** (footprint & template & not a known fault) and on each V1 evaluation zone - not as bounding-box overlap. Output: `tables/dem_coverage.csv`, `dem_catalog/dem_coverage.geojson`, `figures/dem_atlas/dem_coverage.png`.

In [ ]:
P.dem_coverage()
show('tables/dem_coverage.csv')
figs('dem_atlas')

## 06 - Full raw-data and DEM atlas

Native-resolution DEM panels (hillshades by azimuth, slope, curvature, roughness, lineaments, scarps) and the 1 m vs 100 m comparison are produced inside the feasibility atlas (section 07) for the representative cases; the raw 19-band atlas is the G06 one (`figures/raw_atlas` of the G06 run).

In [ ]:
T = show('dem_catalog/dem_inventory.csv', n=0)
if T is not None and len(T):
    display(T.groupby(['status', 'project']).size().rename('tiles').reset_index())

## 07 - DEM feasibility and early go / no-go report (MANDATORY FIRST MILESTONE)

Pilot DEM features on the sources covering the most eligible V1-zone area (label-free choice), fault signal (AUC fault-near vs background, coverage-artefact screen, scale stability, orientation consistency, W2 false-positive support), DEM-score recall vs W2 at the same budget, 20-30 case figures. **Held-out truth is used for this diagnostic only.** Decision rule (predeclared, `feasibility.go_if`): GO / NO_GO / INCONCLUSIVE / BLOCKED_INPUT. Output: `reports/DEM_FEASIBILITY.md`, `tables/dem_fault_signal.csv`, `figures/dem_feasibility/`.

In [ ]:
P.assess_dem_feasibility()
js('dem_catalog/feasibility_decision.json')
p = OUT / 'reports/DEM_FEASIBILITY.md'
if p.exists(): display(Markdown(p.read_text()))
figs('dem_feasibility', 8)

## 08 - Native-resolution DEM feature extraction

Only after a GO (`dem.require_go`). Fine tier at the work resolution (per tile, halo-aware, nodata-aware), coarse tier on one merged 10 m DEM (seamless), exact area-weighted aggregation to 100 m; per-tile products are cached (resume after a disconnect). Output: `dem_features/dem_features_100m.tif`, registry, terrain-vs-geophysics agreement.

In [ ]:
P.prepare_dem_features()
show('dem_features/dem_feature_registry.csv', n=40)

## 09 - Multiscale terrain structure maps

Scarps, lineaments, truncations, drainage alignment and persistence are observations, not interpretations (a scarp is not automatically tectonic).

In [ ]:
st = P.dem_store()
if st is not None:
    from gems_bench.g07 import visualization as V
    f = 8
    names = [n for n in ('dem_scarp_s10', 'dem_lin_s10', 'dem_scarp_persist_fine', 'dem_dircoh_fine', 'dem_scarp_s150', 'dem_valleyline_fine') if n in st.names]
    from gems_bench.g06 import visualization as V6
    V6.plot_feature_atlas({n: st.load([n])[0][::f, ::f] for n in names}, OUT / 'figures' / 'structural_features', 'terrain_structure', 'Terrain structure (100 m aggregates)', None, 1, 3)
    figs('structural_features')
else:
    print('no DEM feature store (DEM_FEATURES not run / not GO)')

## 10 - Existing G06 feature-store restoration

The G06 store is **reused** (staged from Drive, validated: files, alignment, schema hash) - never rebuilt.

In [ ]:
P.restore_g06()
js('dependencies/g06_store.json')

## 11 - Complete training visibility and fold audit

V1 (withheld systems), V3 (geographic), V2 (completion); nested fits `j|k`; explicit isolation tests (written during section 12).

In [ ]:
show('splits/fits.csv', ['fit_id', 'visible_px', 'hidden_px', 'region_px', 'predicts'], n=30)
A = show('splits/isolation_audit.csv', n=0)
if A is not None and len(A): display(A.status.value_counts())

## 12 - Surface patch and pixel datasets

Surface pool on DEM-covered cells, label variants per fit, analysis-aligned surface patches (10 m) for the U-Net; plus the splits audit.

In [ ]:
P.prepare_training_data()
show('datasets/dataset_summary.csv', n=40)
js('datasets/surface/patches/surface_manifest.json', 800)

## 13 - Improved subsurface supervision datasets

Pool drawn with the G06 sampler (denser positives); per fit: G06 weights, IMPROVED weights (system-balanced positives, distance-graded PU, stronger hard negatives, full region balance) and the proximity target exp(-d / 0.3 km).

In [ ]:
L = show('datasets/subsurface/labels/V1_0.parquet', n=0)
if L is not None and len(L):
    display(L.groupby('cls').agg(n=('row_id', 'size'), w_g06=('weight_g06', 'sum'), w_improved=('weight_improved', 'sum'), proximity=('proximity', 'mean')))

## 14 - Surface classical ML benchmarks (G07-A1 / A2)

LightGBM / Random Forest on aggregated DEM channels, DEM-covered cells only; BLOCKED_INPUT without a DEM store.

In [ ]:
P.benchmark_surface_models()
show('models/surface_registry.csv', ['experiment', 'fit_id', 'status', 'kind', 'n_train', 'best_iteration', 'seconds'], n=40)

## 15 - Subsurface improved-supervision benchmarks (G07-B0 / B1 / B2)

Same pool, visibilities and G06-P4 families; only the label design changes.

In [ ]:
P.benchmark_subsurface_models()
show('models/subsurface_registry.csv', ['experiment', 'fit_id', 'status', 'target', 'n_train', 'best_iteration', 'seconds'], n=40)

## 16 - Optional compact surface U-Net (G07-C1)

Only when A1 / A2 show useful evidence. GPU runtime + `CONFIG['use_gpu'] = True`.

In [ ]:
P.train_surface_unet_stage()
show('models/surface_unet_registry.csv')

## 17 - Full candidate inventory (label-free pools)

W2, CNN, surface, subsurface, magnetic, gravity, structural, persistence pools (exact-k), merged and de-duplicated with provenance; candidate recall before ranking (G07-C2).

In [ ]:
P.generate_candidates()
show('candidates/candidate_summary.csv', ['unit', 'n_candidates', 'pools'])
R = show('evaluation/recall/candidate_recall.csv', n=0)
if R is not None and len(R):
    display(R.groupby(['scenario', 'source']).agg(systems=('n_systems', 'sum'), covered=('n_covered', 'sum'), area=('area_frac', 'mean')))

## 18 - Candidate feature engineering

Geometry, G06 geology + DEM means and steps, orientation agreement, permitted known context (nested for training units), honest model evidence, expert agreement, uncertainty; synthetic positives / negatives from TRAINING truth only.

In [ ]:
D = P._load_candidates()
if len(D):
    display(D.groupby(['unit', 'synthetic']).agg(n=('label', 'size'), relevant=('target', 'sum')))
    from gems_bench.g07.models_candidates import column_family
    display(pd.Series([column_family(c) for c in D.columns]).value_counts())

## 19 - Geological candidate classification and ranking (G07-D1)

Cross-fitted over units; isotonic calibration from other units' held-out scores; utility; portfolio (D2) and corridor (E1) maps.

In [ ]:
P.train_candidate_ranker()
js('models/candidate_registry.json', 2500)

## 20 - Expert complementarity analysis

Systems each expert recovers that W2 does not, at the same exact budget (filled by section 22).

In [ ]:
show('evaluation/metrics/complementarity.csv', n=30)

## 21 - DTI-aware exact-budget selection

Every top-area selection has exactly k eligible pixels (deterministic label-free ties); the audit shows what the legacy `>=` rule would have selected.

In [ ]:
P.evaluate()
show('evaluation/metrics/exact_budget.csv')
figs('model_comparison', 3)

## 22 - V1 complete-system discovery evaluation (primary)

In [ ]:
f = cfg['representation']['primary_frac']
S = show('evaluation/metrics/metrics.csv', n=0)
if S is not None and len(S):
    display(S[(S.scenario == 'V1') & (S.frac == f)][['model', 'rep', 'strictness', 'dti', 'worst_zone_dti', 'tp_w', 'fp_w', 'fn_w', 'area_frac_eligible']].sort_values('dti', ascending=False))
show('evaluation/metrics/model_comparison.csv', n=30)

## 23 - V3 geographic evaluation

In [ ]:
show('evaluation/geographic/v3_summary.csv', ['model', 'rep', 'frac', 'dti', 'worst_zone_dti', 'zone_dtis'], n=30)
figs('dti_frontiers', 3)

## 24 - V2 completion diagnostics

In [ ]:
if S is not None and len(S):
    display(S[S.scenario == 'V2'][['model', 'rep', 'frac', 'dti', 'worst_zone_dti']].sort_values(['frac', 'dti']))

## 25 - Recovered / lost fault-system atlas

Truth outlines are post-hoc only.

In [ ]:
show('evaluation/metrics/system_recovery.csv', ['model', 'unit', 'rep', 'frac', 'n_systems', 'n_recovered', 'n_recovered_w2', 'n_new', 'n_lost', 'new_ids', 'lost_ids'], n=40)
figs('recovery_atlas')

## 26 - Feature / model ablations

In [ ]:
show('evaluation/ablations/ablations.csv', n=30)
show('evaluation/promotion/promotion_decisions.csv', n=20)

## 27 - Full-footprint prediction reconstruction

W2 base = frozen P6 full-region score; the promoted policy (or the W2 exact-k reference) on the competition budget domain; parity checks.

In [ ]:
P.full_inference()
js('predictions/full_region/full_region_manifest.json', 2500)

## 28 - Export and submission parity checks

Gates: EXPORT_SUBMISSION, auto_submit disabled, promoted strict model, full-region parity, DEM competition eligibility (when the DEM is used), provenance. At best READY_FOR_MANUAL_UPLOAD; never uploaded. W2 is retained when anything fails.

In [ ]:
P.export_submission()
js('evaluation/promotion/submission_eligibility.json')

## 29 - Comprehensive scientific report

In [ ]:
P.build_report()
for r in ('DEM_FEASIBILITY', 'DTI_EVALUATION', 'COMPLEMENTARITY', 'SUBMISSION_ELIGIBILITY', 'NEXT_EXPERIMENT'):
    p = OUT / 'reports' / f'{r}.md'
    if p.exists(): display(Markdown(p.read_text()))

## 30 - Artifact inventory and resume guide

Resume: re-run 01-02 + init with the same CONFIG; validated stages are REUSED, DEM tiles and models are cached by fingerprint. `P.run()` runs everything with the same RUN gates.

In [ ]:
display(P.stage_table())
inv = show('manifests/artifact_inventory.csv', n=0)
if inv is not None and len(inv):
    display(inv.groupby('section').agg(files=('path', 'size'), mb=('bytes', lambda b: round(b.sum() / 1e6, 1))))
P.final_message()
# P.run()